# Local UITI_VANO Interpretability

Notebook-first workflow for steps 1-3: circuit/date selection, deterministic critical-point detection, structured context, and optional LLM interpretation.

## 1. Setup and parameters

In [1]:
%load_ext autoreload
%autoreload 2

import json
import os
import sys
from datetime import datetime, timezone
from pathlib import Path

try:
    from dotenv import load_dotenv
except ImportError:
    load_dotenv = None

# 1. Dynamically locate the project root by finding the 'src' directory
ROOT = Path.cwd().resolve()
while not (ROOT / "src").is_dir() and ROOT.parent != ROOT:
    ROOT = ROOT.parent

# 2. Add 'src' to sys.path so Python can find 'chec_local_interpreter'
src_path = str(ROOT / "src")
if src_path not in sys.path:
    sys.path.insert(0, src_path)

# 3. Load environment variables
if load_dotenv:
    load_dotenv(ROOT / ".env")

# 4. Import project modules
from chec_local_interpreter.attribution import enrich_critical_points
from chec_local_interpreter.config import CriticalityThresholds, DEFAULT_DATA_PATH, MAX_CRITICAL_POINTS
from chec_local_interpreter.context_builder import build_context_package, critical_points_frame, save_json_artifact
from chec_local_interpreter.critical_points import build_daily_series, compute_daily_features, detect_critical_periods, detect_point_reasons, rank_critical_points
from chec_local_interpreter.data_loader import available_circuits, dataset_summary, filter_events, load_dataset, resolve_columns
from chec_local_interpreter.llm_client import call_llm, save_cot_html_graph
from chec_local_interpreter.llm_contracts import PROMPT_VERSION, load_output_schema, render_prompt, save_prompt_artifact
from chec_local_interpreter.llm_skills import assemble_skill_bundle, list_available_skills, verify_required_skills
from chec_local_interpreter.llm_validation import save_invalid_output, validate_llm_response
from chec_local_interpreter.plotting import save_uiti_vano_plot, plot_interactive_critical_points



DATA_PATH = str(ROOT / "data/Indicadores_vano_v3.csv")


## 2. Load data

In [2]:
import pandas as pd
raw_df = load_dataset(DATA_PATH)
column_resolution = resolve_columns(raw_df)
summary = dataset_summary(raw_df)
available = available_circuits(raw_df)
print(f"Dataset shape: {summary['shape']}")
print(f"Date range: {summary['date_min']} to {summary['date_max']}")
print(f"Available circuits: {summary['available_circuits_count']}")
print(f"Selected circuits parameter: {'all circuits'}")
print(f"Unavailable optional columns: {len(column_resolution.unavailable_optional)}")

Dataset shape: [159470, 273]
Date range: 2025-11-01 to 2026-04-30
Available circuits: 208
Selected circuits parameter: all circuits
Unavailable optional columns: 0


In [3]:
raw_df.head()

,CIRCUITO,FID_SW,COD_EQ_PROTEGE,FID_VANO,T_USUS_EQ_PROT,LVSW,CNT_VN,CNT_VN_SW,FECHA,DURACION,...,clouds_15,clouds_16,clouds_17,clouds_18,clouds_19,clouds_20,clouds_21,clouds_22,clouds_23,clouds_24
0,AGU23L14,20142930,AGU23L14,20139439,1821,2.568,16,22,2025-11-03 23:56:04,0.008,...,100.0,100.0,100.0,100.0,100.0,100.0,100.0,100.0,100.0,100.0
1,AGU23L14,20142930,AGU23L14,20139439,1821,2.568,16,22,2025-11-18 17:08:14,0.008,...,100.0,100.0,84.0,99.0,99.0,98.0,98.0,97.0,98.0,100.0
2,AGU23L14,20142930,AGU23L14,20139439,1821,2.568,16,22,2025-11-19 20:25:50,0.008,...,79.0,99.0,100.0,100.0,100.0,97.0,100.0,100.0,99.0,98.0
3,AGU23L14,20142930,AGU23L14,20139439,1821,2.568,16,22,2025-12-10 21:33:41,0.008,...,69.0,93.0,100.0,55.0,61.0,32.0,56.0,59.0,58.0,76.0
4,AGU23L14,20142930,AGU23L14,20139439,1821,2.568,16,22,2025-12-19 15:59:10,0.041,...,99.0,97.0,100.0,100.0,100.0,100.0,100.0,100.0,100.0,100.0


## 2.1 Circuit Histograms: number of events and UITI

In [4]:
from chec_local_interpreter.plotting import plot_interactive_circuit_events

# =======================================================
# EJEMPLOS DE USO:
# =======================================================

# 1. Utilizar todo el intervalo de datos (sin filtrar)
fig = plot_interactive_circuit_events(raw_df)
fig.show()

# 2. Utilizar un intervalo de fechas específico
#fig = plot_interactive_circuit_events(raw_df, start_date='2025-11-01 00:00:01', end_date='2025-12-01 00:00:01')
#fig.show()


In [5]:
from chec_local_interpreter.plotting import plot_interactive_uiti_vano_sums

# =======================================================
# EJEMPLOS DE USO:
# =======================================================

# 1. Utilizar todo el intervalo de datos (sin filtrar)
fig_sums = plot_interactive_uiti_vano_sums(raw_df)
fig_sums.show()

# 2. Utilizar un intervalo de fechas específico
# fig_sums = plot_interactive_uiti_vano_sums(raw_df, start_date='2025-11-01 00:00:01', end_date='2025-12-01 00:00:01')
# fig_sums.show()


In [6]:
from chec_local_interpreter.plotting import run_kmeans, plot_interactive_circuit_clustering

# =======================================================
# EJEMPLOS DE USO:
# =======================================================

# Lista de circuitos a destacar (opcional)
circuitos_destacados = [
     'CHA23L18', 'CHA23L14', 'BOA23L13', 'VBO23L14', 
     'BUM23L12', 'VBO23L15', 'SIO23L15', 'BOA23L12', 
     'AMA23L12', 'QHI23L13', 'IRR23L13', 'RIO23L14', 'DON23L12','DON23L13'
 ]

# 1. Utilizar todo el intervalo de datos con los destacados:
fig_cluster = plot_interactive_circuit_clustering(raw_df, highlighted_circuits=circuitos_destacados)
fig_cluster.show()

# 2. Utilizar un intervalo de fechas y sin circuitos destacados:
# fig_cluster2 = plot_interactive_circuit_clustering(raw_df, start_date='2025-11-01 00:00:01', end_date='2025-12-01 00:00:01')
# fig_cluster2.show()


## 2.2 GIS MAP

In [8]:
from chec_local_interpreter.plotting import plot_circuit_map_plotly

# =================================================================
# Ejemplo de uso:
# =================================================================
plot_circuit_map_plotly(
    df=raw_df, 
    circuito_name='DON23L13', 
    date_range=None, 
    color_target='number_of_events'
)




In [9]:
plot_circuit_map_plotly(
    df=raw_df, 
    circuito_name='DON23L13', 
    date_range=None, 
    color_target='UITI_VANO_sum'
)

## 3. Filter dataset

In [22]:
SELECTED_CIRCUITOS = ['DON23L13']
START_DATE = pd.to_datetime(raw_df['FECHA']).min()
END_DATE = pd.to_datetime(raw_df['FECHA']).max()
MAX_CRITICAL_POINTS = 5
OUTPUT_DIR = Path("outputs")
CALL_LLM = True
import os
from dotenv import load_dotenv
load_dotenv()
LLM_MODEL = os.getenv("LLM_MODEL", "Desconocido")
LLM_PROVIDER = os.getenv("LLM_PROVIDER", "ollama")

timestamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
DATA_PATH = str((ROOT / DATA_PATH).resolve() if not Path(DATA_PATH).is_absolute() else Path(DATA_PATH))
DATA_PATH

'/Users/andresalvarez/Documents/chec-local-uiti-vano-interpreter/data/Indicadores_vano_v3.csv'

In [11]:
events_df = filter_events(
    raw_df,
    selected_circuitos=SELECTED_CIRCUITOS,
    start_date=START_DATE,
    end_date=END_DATE,
)
selected_circuitos_effective = SELECTED_CIRCUITOS or sorted(events_df['CIRCUITO'].dropna().astype(str).unique().tolist())
start_effective = START_DATE or (events_df['fecha_dia'].min().date().isoformat() if not events_df.empty else None)
end_effective = END_DATE or (events_df['fecha_dia'].max().date().isoformat() if not events_df.empty else None)
print(f"Filtered rows: {len(events_df)}")
print(f"Effective circuits: {selected_circuitos_effective[:10]}{' ...' if len(selected_circuitos_effective) > 10 else ''}")
print(f"Effective window: {start_effective} to {end_effective}")

Filtered rows: 9418
Effective circuits: ['DON23L13']
Effective window: 2025-11-01 00:00:01 to 2026-04-30 21:53:40


In [12]:
events_df.head()

,CIRCUITO,FID_SW,COD_EQ_PROTEGE,FID_VANO,T_USUS_EQ_PROT,LVSW,CNT_VN,CNT_VN_SW,FECHA,DURACION,...,clouds_16,clouds_17,clouds_18,clouds_19,clouds_20,clouds_21,clouds_22,clouds_23,clouds_24,fecha_dia
6010,DON23L13,20477862,L13913,20475320,2,1.192,4,7,2026-01-31 18:34:00,14.183,...,100.0,100.0,100.0,100.0,100.0,100.0,99.0,99.0,98.0,2026-01-31
6011,DON23L13,20477862,L13913,20475320,2,1.192,4,7,2026-03-03 15:04:00,3.967,...,60.0,100.0,98.0,100.0,99.0,99.0,91.0,49.0,67.0,2026-03-03
6012,DON23L13,20477862,L13913,20475320,2,1.192,4,7,2026-04-20 14:50:00,2.0,...,100.0,100.0,100.0,100.0,99.0,96.0,100.0,100.0,98.0,2026-04-20
6013,DON23L13,20477862,L13913,20475805,2,1.609,6,7,2026-01-31 18:34:00,14.183,...,100.0,100.0,100.0,100.0,100.0,100.0,99.0,99.0,98.0,2026-01-31
6014,DON23L13,20477862,L13913,20475805,2,1.609,6,7,2026-03-03 15:04:00,3.967,...,60.0,100.0,98.0,100.0,99.0,99.0,91.0,49.0,67.0,2026-03-03


## 4. Build daily UITI_VANO series

In [13]:
daily_df = build_daily_series(events_df)
display(daily_df.head())
print(f"Daily rows: {len(daily_df)}")
print(f"Total UITI_VANO: {daily_df['UITI_VANO'].sum() if not daily_df.empty else 0}")

,fecha_dia,UITI_VANO,event_count,DURACION_total,users_total,UITI,DURACION_available,users_available,UITI_available
0,2025-11-01,244.812744,166,20.537,160339,516.592,True,True,True
1,2025-11-02,15.633321,4,26.732,4,26.732,True,True,True
2,2025-11-03,962.386255,122,195.244,3429,1651.989,True,True,True
3,2025-11-04,0.000000,0,0.000,0,0.000,False,False,False
4,2025-11-05,774.735923,100,148.280,5302,1346.386,True,True,True


Daily rows: 181
Total UITI_VANO: 283732.8099875111


In [14]:
daily_df['event_count'].sum()

np.int64(9418)

## 5. Detect relevant/critical points

In [15]:
thresholds = CriticalityThresholds(max_points=MAX_CRITICAL_POINTS)
feature_df = compute_daily_features(daily_df, metric="UITI_VANO")
reasons = detect_point_reasons(feature_df, thresholds=thresholds, metric="UITI_VANO")
critical_points = rank_critical_points(feature_df, reasons, max_points=MAX_CRITICAL_POINTS, metric="UITI_VANO")
critical_periods = detect_critical_periods(feature_df, thresholds=thresholds, metric="UITI_VANO")
display(critical_points_frame(critical_points))
critical_periods

,critical_point_id,fecha_dia,rank,criticality_score,criticality_types,selection_reason,UITI_VANO,event_count
0,cp-2025-12-01,2025-12-01,1,10.0,high_robust_z;local_peak;sharp_positive_change...,UITI_VANO sube bruscamente frente al dia anter...,84931.4014,584
1,cp-2026-02-02,2026-02-02,2,10.0,high_robust_z;local_peak;sharp_positive_change,UITI_VANO sube bruscamente frente al dia anter...,17176.6106,195
2,cp-2025-11-16,2025-11-16,3,10.0,high_robust_z;local_peak;sharp_positive_change,UITI_VANO sube bruscamente frente al dia anter...,14561.9872,175
3,cp-2026-04-15,2026-04-15,4,10.0,high_robust_z;local_peak;sharp_positive_change,UITI_VANO sube bruscamente frente al dia anter...,13292.9001,152
4,cp-2026-03-02,2026-03-02,5,10.0,high_robust_z;local_peak;sharp_positive_change,UITI_VANO sube bruscamente frente al dia anter...,13067.5763,757
5,cp-2026-02-17,2026-02-17,6,10.0,high_robust_z;local_peak;sharp_positive_change,UITI_VANO sube bruscamente frente al dia anter...,10996.4654,176
6,cp-2026-04-22,2026-04-22,7,10.0,high_robust_z;local_peak;sharp_positive_change,UITI_VANO sube bruscamente frente al dia anter...,9881.7326,114
7,cp-2026-02-06,2026-02-06,8,10.0,high_robust_z;local_peak;sharp_positive_change,UITI_VANO sube bruscamente frente al dia anter...,8904.4020,193
8,cp-2026-02-13,2026-02-13,9,10.0,high_robust_z;sharp_positive_change,UITI_VANO sube bruscamente frente al dia anter...,8389.3340,695
9,cp-2025-11-23,2025-11-23,10,10.0,sharp_positive_change,UITI_VANO sube bruscamente frente al dia anter...,7581.9436,198


[{'critical_period_id': 'period-2025-11-29-2025-12-01',
  'start_date': '2025-11-29',
  'end_date': '2025-12-01',
  'days': 3,
  'period_type': 'sustained_elevated_uiti_vano',
  'score': 0.3095,
  'total_uiti_vano': 87825.8447,
  'summary': 'UITI_VANO permanecio elevado durante 3 dias entre 2025-11-29 y 2025-12-01.'},
 {'critical_period_id': 'period-2026-02-13-2026-02-17',
  'start_date': '2026-02-13',
  'end_date': '2026-02-17',
  'days': 5,
  'period_type': 'sustained_elevated_uiti_vano',
  'score': 0.1374,
  'total_uiti_vano': 38992.3369,
  'summary': 'UITI_VANO permanecio elevado durante 5 dias entre 2026-02-13 y 2026-02-17.'},
 {'critical_period_id': 'period-2026-02-28-2026-03-02',
  'start_date': '2026-02-28',
  'end_date': '2026-03-02',
  'days': 3,
  'period_type': 'sustained_elevated_uiti_vano',
  'score': 0.0797,
  'total_uiti_vano': 22605.0033,
  'summary': 'UITI_VANO permanecio elevado durante 3 dias entre 2026-02-28 y 2026-03-02.'},
 {'critical_period_id': 'period-2026-01-

## 6. Enrich critical points

In [16]:
critical_points = enrich_critical_points(events_df, critical_points)
print(f"Enriched critical points: {len(critical_points)}")
critical_points[:1]

Enriched critical points: 12


[{'critical_point_id': 'cp-2025-12-01',
  'fecha_dia': '2025-12-01',
  'criticality_score': 10.0,
  'criticality_types': ['high_robust_z',
   'local_peak',
   'sharp_positive_change',
   'top_contribution_day'],
  'selection_reason': 'UITI_VANO sube bruscamente frente al dia anterior. El dia aporta una fraccion alta del UITI_VANO total de la ventana.',
  'reasons': [{'reason_type': 'sharp_positive_change',
    'score': 171.6543,
    'value': 82986.935,
    'threshold': 3.0,
    'detail': 'UITI_VANO sube bruscamente frente al dia anterior.'},
   {'reason_type': 'top_contribution_day',
    'score': 2.9934,
    'value': 84931.4014,
    'threshold': 0.1,
    'detail': 'El dia aporta una fraccion alta del UITI_VANO total de la ventana.'},
   {'reason_type': 'high_robust_z',
    'score': 2.0,
    'value': 84931.4014,
    'threshold': 3.0,
    'detail': 'UITI_VANO esta por encima de la linea base robusta de la ventana.'},
   {'reason_type': 'local_peak',
    'score': 0.2993,
    'value': 8493

In [16]:
fig_critical = plot_interactive_critical_points(
    daily_df,
    critical_points,
    selected_circuitos=selected_circuitos_effective,
    start_date=start_effective,
    end_date=end_effective,
)
fig_critical.show()

## 7. Build structured context JSON

In [ ]:
context_package = build_context_package(
    raw_df=raw_df,
    events_df=events_df,
    daily_df=daily_df,
    critical_points=critical_points,
    critical_periods=critical_periods,
    selected_circuitos=selected_circuitos_effective,
    start_date=start_effective,
    end_date=end_effective,
)
print(json.dumps({
    'analysis_name': context_package['analysis_name'],
    'selected_context': context_package['selected_context'],
    'summary': context_package['summary'],
    'critical_point_count': len(context_package['critical_points']),
}, ensure_ascii=False, indent=2))

{
  "analysis_name": "local_uiti_vano_interpretability",
  "selected_context": {
    "circuitos": [
      "DON23L13"
    ],
    "start_date": "2025-11-01",
    "end_date": "2026-04-30",
    "indicator": "UITI_VANO"
  },
  "window_summary": {
    "row_count": 9418,
    "event_count": 9418,
    "nonzero_days": 79,
    "total_uiti_vano": 283732.81,
    "max_uiti_vano_date": "2025-12-01",
    "max_uiti_vano_value": 84931.4014
  },
  "critical_point_count": 12
}


## 8. Save artifacts

In [19]:
context_path = save_json_artifact(context_package, OUTPUT_DIR / f"structured_context_{timestamp}.json")
critical_csv_path = OUTPUT_DIR / f"critical_points_{timestamp}.csv"
critical_points_frame(critical_points).to_csv(critical_csv_path, index=False)
plot_path = save_uiti_vano_plot(
    daily_df,
    critical_points,
    selected_circuitos=selected_circuitos_effective,
    start_date=start_effective,
    end_date=end_effective,
    output_path=OUTPUT_DIR / f"uiti_vano_timeseries_{timestamp}.png",
)
print(context_path)
print(critical_csv_path)
print(plot_path)

Matplotlib is building the font cache; this may take a moment.


outputs/structured_context_20260615T022127Z.json
outputs/critical_points_20260615T022127Z.csv
outputs/uiti_vano_timeseries_20260615T022127Z.png


## 9. LLM skills, contracts, and validation

In [ ]:
import subprocess
import time
import urllib.request
import urllib.error

def ensure_ollama_and_model(model_name=LLM_MODEL):
    print(f"🔍 Checking Ollama installation...")
    try:
        subprocess.run(["ollama", "--version"], check=True, capture_output=True)
    except (subprocess.CalledProcessError, FileNotFoundError):
        print("⚠️ Ollama is not installed or not in PATH. Please install it first: brew install --cask ollama")
        return False

    print("🔍 Checking if Ollama server is running...")
    server_running = False
    try:
        req = urllib.request.Request("http://localhost:11434/")
        with urllib.request.urlopen(req, timeout=2) as response:
            if response.getcode() == 200:
                server_running = True
    except (urllib.error.URLError, ConnectionError):
        pass

    if not server_running:
        print("🚀 Starting Ollama server in the background...")
        subprocess.Popen(["ollama", "serve"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
        time.sleep(3)

    print(f"📥 Ensuring model {model_name} is downloaded. This may take a while if it is the first time...")
    try:
        subprocess.run(["ollama", "pull", model_name], check=True)
        print("✅ Model is ready!")
        return True
    except subprocess.CalledProcessError as e:
        print(f"❌ Failed to pull the model: {e}")
        return False

if CALL_LLM and LLM_PROVIDER == "ollama":
    ensure_ollama_and_model(LLM_MODEL)


In [20]:
missing_skills = verify_required_skills()
if missing_skills:
    raise FileNotFoundError(f"Missing required LLM skills: {missing_skills}")
skills = list_available_skills()
skill_bundle = assemble_skill_bundle()
output_schema = load_output_schema()
prompt = render_prompt(
    context_json=json.dumps(context_package, ensure_ascii=False),
    output_schema_json=json.dumps(output_schema, ensure_ascii=False),
    prompt_version=PROMPT_VERSION,
)
prompt_path = save_prompt_artifact(prompt, OUTPUT_DIR / f"llm_prompt_{timestamp}.md")
print(f"Loaded skills: {skills}")
print(f"Prompt saved: {prompt_path}")
print(f"Skill bundle characters: {len(skill_bundle)}")

Loaded skills: ['01_structured_context_builder.md', '02_critical_point_interpreter.md', '03_uiti_vano_behavior_explainer.md', '04_domain_grounding_guardrails.md', '05_llm_output_validator.md']
Prompt saved: outputs/llm_prompt_20260615T022127Z.md
Skill bundle characters: 4832


In [23]:
llm_result = call_llm(
    prompt,
    provider=LLM_PROVIDER,
    model=LLM_MODEL,
    call_enabled=CALL_LLM,
)
print(llm_result.message)
if llm_result.think_content:
    circuit_name_str = ", ".join(selected_circuitos_effective) if selected_circuitos_effective else "Todos"
    graph_path = save_cot_html_graph(llm_result.think_content, OUTPUT_DIR / f"cot_graph_{timestamp}.html", circuit_name=circuit_name_str)
    print(f"\n>>> CoT graph guardado en la ruta de outputs: {graph_path}")

if llm_result.output_text:
    import json
    import re
    try:
        json_str = llm_result.output_text
        # Buscar delimitadores JSON si existen
        match = re.search(r'```json\s*(.*?)\s*```', json_str, re.DOTALL)
        if match:
            json_str = match.group(1)
        else:
            # Intento robusto de encontrar el bloque JSON
            start_idx = json_str.find('{')
            end_idx = json_str.rfind('}')
            if start_idx != -1 and end_idx != -1:
                json_str = json_str[start_idx:end_idx+1]
        
        validation_data = json.loads(json_str)
        analysis_path = save_json_artifact(validation_data, OUTPUT_DIR / f"llm_analysis_{timestamp}.json")
        print(f"\n>>> Raw LLM analysis JSON saved: {analysis_path}")
        display(validation_data)
    except Exception as e:
        print(f"Failed to parse JSON from LLM: {e}. Construyendo diccionario de respaldo para forzar renderizado.")
        # Diccionario de respaldo para asegurar que SIEMPRE se pueda generar el HTML
        validation_data = {
            "headline": "Error en el formato de respuesta del LLM",
            "executive_summary": "El modelo no retornó un JSON válido para procesar automáticamente.",
            "period_synthesis": f"Respuesta cruda del modelo:\n\n{llm_result.output_text}"
        }


OPENAI_API_KEY is not configured; prompt saved for manual use.


# 10 Html results visualization

In [23]:
from chec_local_interpreter.plotting import render_llm_analysis

# Renderizando directamente asumiendo JSON parseado en validation_data (sin validación estricta):
if 'validation_data' in locals() and validation_data:
    html_path = render_llm_analysis(
        validation_data=validation_data,
        raw_df=raw_df,
        daily_df=daily_df,
        critical_points=critical_points,
        selected_circuitos=selected_circuitos_effective,
        start_date=start_effective,
        end_date=end_effective,
        llm_model=LLM_MODEL,
        llm_provider=LLM_PROVIDER
    )
    print(f"\n>>> HTML Analysis Report guardado en la ruta de outputs: {html_path}")
else:
    print("Por favor ejecuta primero la celda del LLM y asegúrate de que haya generado validation_data.")


Por favor ejecuta primero la celda del LLM y asegúrate de que sea válida.
